In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier


In [2]:
data_path = "StudentPerformance/data/Student_performance_data _.csv"
df = pd.read_csv(data_path)
print(df.columns.tolist())

['StudentID', 'Age', 'Gender', 'Ethnicity', 'ParentalEducation', 'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering', 'GPA', 'GradeClass']


In [3]:

target_col = "GradeClass"
drop_cols = ["StudentID"]  # ID nuk e perdorim si feature

X = df.drop(columns=drop_cols + [target_col])
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target value counts:")
print(y.value_counts())


X shape: (2392, 13)
y shape: (2392,)
Target value counts:
GradeClass
4.0    1211
3.0     414
2.0     391
1.0     269
0.0     107
Name: count, dtype: int64


In [4]:
label_enc = LabelEncoder()
y_encoded = label_enc.fit_transform(y)
print("Classes:", label_enc.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Classes: [0. 1. 2. 3. 4.]
Train shape: (1913, 13)
Test shape: (479, 13)


In [5]:

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

Numeric features: ['Age', 'Gender', 'Ethnicity', 'ParentalEducation', 'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering', 'GPA']
Categorical features: []


In [6]:

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),                 # i njejte per krejt klasifikuesit
    ("feature_selection", SelectKBest(score_func=f_classif)),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [7]:

rf_param_grid = {
    "feature_selection__k": [3, 5, 7, 10, "all"],     # sa veçori me mbajt

    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 5, 10, 20],
    "classifier__max_features": ["sqrt", "log2"],
}

In [8]:
# Train (if needed), evaluate and save Random Forest results
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

try:
    # Ensure training data and preprocessor exist
    if 'X_train' not in globals() or 'y_train' not in globals():
        raise NameError('Training data not found. Run preprocessing cells first.')

    if 'rf_grid_search' not in globals():
        rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])
        param_grid = {
            'classifier__n_estimators': [100],
            'classifier__max_depth': [None, 10]
        }
        rf_grid_search = GridSearchCV(rf_pipeline, param_grid, scoring='f1_macro', cv=5, n_jobs=-1)
        print('Fitting Random Forest grid search (this may take a while)...')
        rf_grid_search.fit(X_train, y_train)

    rf_best = rf_grid_search.best_estimator_
    rf_y_pred = rf_best.predict(X_test)

    # Metrics
    rf_accuracy = accuracy_score(y_test, rf_y_pred)
    rf_precision = precision_score(y_test, rf_y_pred, average='macro')
    rf_recall = recall_score(y_test, rf_y_pred, average='macro')
    rf_f1 = f1_score(y_test, rf_y_pred, average='macro')

    print('
Random Forest - Test Metrics:')
    print(f"Accuracy: {rf_accuracy:.4f}")
    print(f"Precision (macro): {rf_precision:.4f}")
    print(f"Recall (macro): {rf_recall:.4f}")
    print(f"F1-score (macro): {rf_f1:.4f}")

    print('
Random Forest - Classification Report:')
    print(classification_report(y_test, rf_y_pred, target_names=[str(c) for c in label_enc.classes_]))

    # Confusion matrix
    rf_cm = confusion_matrix(y_test, rf_y_pred)
    os.makedirs('StudentPerformance/visualizations', exist_ok=True)
    plt.figure(figsize=(7,5))
    sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=label_enc.classes_, yticklabels=label_enc.classes_)
    plt.title('Confusion Matrix - Random Forest')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig('StudentPerformance/visualizations/random_forest_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Save results
    rf_results_df = pd.DataFrame({
        'Model': ['Random Forest'],
        'Best_Params': [str(rf_grid_search.best_params_)],
        'CV_F1_Macro': [rf_grid_search.best_score_],
        'Test_Accuracy': [rf_accuracy],
        'Test_Precision_Macro': [rf_precision],
        'Test_Recall_Macro': [rf_recall],
        'Test_F1_Macro': [rf_f1]
    })
    os.makedirs('StudentPerformance/results', exist_ok=True)
    rf_results_df.to_csv('StudentPerformance/results/random_forest_results.csv', index=False)
    print('Results saved to StudentPerformance/results/random_forest_results.csv')

except Exception as e:
    print('Error running Random Forest evaluation:', e)


SyntaxError: unterminated string literal (detected at line 29) (2672413607.py, line 29)

In [9]:
# Train (if needed), evaluate and save Random Forest results
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

try:
    # Ensure training data and preprocessor exist
    if 'X_train' not in globals() or 'y_train' not in globals():
        raise NameError('Training data not found. Run preprocessing cells first.')

    if 'rf_grid_search' not in globals():
        rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])
        param_grid = {
            'classifier__n_estimators': [100],
            'classifier__max_depth': [None, 10]
        }
        rf_grid_search = GridSearchCV(rf_pipeline, param_grid, scoring='f1_macro', cv=5, n_jobs=-1)
        print('Fitting Random Forest grid search (this may take a while)...')
        rf_grid_search.fit(X_train, y_train)

    rf_best = rf_grid_search.best_estimator_
    rf_y_pred = rf_best.predict(X_test)

    # Metrics
    rf_accuracy = accuracy_score(y_test, rf_y_pred)
    rf_precision = precision_score(y_test, rf_y_pred, average='macro')
    rf_recall = recall_score(y_test, rf_y_pred, average='macro')
    rf_f1 = f1_score(y_test, rf_y_pred, average='macro')

    print('
Random Forest - Test Metrics:')
    print(f"Accuracy: {rf_accuracy:.4f}")
    print(f"Precision (macro): {rf_precision:.4f}")
    print(f"Recall (macro): {rf_recall:.4f}")
    print(f"F1-score (macro): {rf_f1:.4f}")

    print('
Random Forest - Classification Report:')
    print(classification_report(y_test, rf_y_pred, target_names=[str(c) for c in label_enc.classes_]))

    # Confusion matrix
    rf_cm = confusion_matrix(y_test, rf_y_pred)
    os.makedirs('StudentPerformance/visualizations', exist_ok=True)
    plt.figure(figsize=(7,5))
    sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=label_enc.classes_, yticklabels=label_enc.classes_)
    plt.title('Confusion Matrix - Random Forest')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig('StudentPerformance/visualizations/random_forest_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Save results
    rf_results_df = pd.DataFrame({
        'Model': ['Random Forest'],
        'Best_Params': [str(rf_grid_search.best_params_)],
        'CV_F1_Macro': [rf_grid_search.best_score_],
        'Test_Accuracy': [rf_accuracy],
        'Test_Precision_Macro': [rf_precision],
        'Test_Recall_Macro': [rf_recall],
        'Test_F1_Macro': [rf_f1]
    })
    os.makedirs('StudentPerformance/results', exist_ok=True)
    rf_results_df.to_csv('StudentPerformance/results/random_forest_results.csv', index=False)
    print('Results saved to StudentPerformance/results/random_forest_results.csv')

except Exception as e:
    print('Error running Random Forest evaluation:', e)


SyntaxError: unterminated string literal (detected at line 29) (2672413607.py, line 29)

In [10]:

rf_accuracy = accuracy_score(y_test, rf_y_pred)
rf_precision = precision_score(y_test, rf_y_pred, average="macro")
rf_recall = recall_score(y_test, rf_y_pred, average="macro")
rf_f1 = f1_score(y_test, rf_y_pred, average="macro")

print("\nRandom Forest - Test Metrics:")
print(f"Accuracy: {rf_accuracy:.4f}")
print(f"Precision (macro): {rf_precision:.4f}")
print(f"Recall (macro): {rf_recall:.4f}")
print(f"F1-score (macro): {rf_f1:.4f}")

print("\nRandom Forest - Classification Report:")
print(classification_report(y_test, rf_y_pred, target_names=[str(c) for c in label_enc.classes_]))

NameError: name 'rf_y_pred' is not defined

In [11]:
# Train (if needed), evaluate and save Random Forest results
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

try:
    # Ensure training data and preprocessor exist
    if 'X_train' not in globals() or 'y_train' not in globals():
        raise NameError('Training data not found. Run preprocessing cells first.')

    if 'rf_grid_search' not in globals():
        rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])
        param_grid = {
            'classifier__n_estimators': [100],
            'classifier__max_depth': [None, 10]
        }
        rf_grid_search = GridSearchCV(rf_pipeline, param_grid, scoring='f1_macro', cv=5, n_jobs=-1)
        print('Fitting Random Forest grid search (this may take a while)...')
        rf_grid_search.fit(X_train, y_train)

    rf_best = rf_grid_search.best_estimator_
    rf_y_pred = rf_best.predict(X_test)

    # Metrics
    rf_accuracy = accuracy_score(y_test, rf_y_pred)
    rf_precision = precision_score(y_test, rf_y_pred, average='macro')
    rf_recall = recall_score(y_test, rf_y_pred, average='macro')
    rf_f1 = f1_score(y_test, rf_y_pred, average='macro')

    print('
Random Forest - Test Metrics:')
    print(f"Accuracy: {rf_accuracy:.4f}")
    print(f"Precision (macro): {rf_precision:.4f}")
    print(f"Recall (macro): {rf_recall:.4f}")
    print(f"F1-score (macro): {rf_f1:.4f}")

    print('
Random Forest - Classification Report:')
    print(classification_report(y_test, rf_y_pred, target_names=[str(c) for c in label_enc.classes_]))

    # Confusion matrix
    rf_cm = confusion_matrix(y_test, rf_y_pred)
    os.makedirs('StudentPerformance/visualizations', exist_ok=True)
    plt.figure(figsize=(7,5))
    sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=label_enc.classes_, yticklabels=label_enc.classes_)
    plt.title('Confusion Matrix - Random Forest')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig('StudentPerformance/visualizations/random_forest_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Save results
    rf_results_df = pd.DataFrame({
        'Model': ['Random Forest'],
        'Best_Params': [str(rf_grid_search.best_params_)],
        'CV_F1_Macro': [rf_grid_search.best_score_],
        'Test_Accuracy': [rf_accuracy],
        'Test_Precision_Macro': [rf_precision],
        'Test_Recall_Macro': [rf_recall],
        'Test_F1_Macro': [rf_f1]
    })
    os.makedirs('StudentPerformance/results', exist_ok=True)
    rf_results_df.to_csv('StudentPerformance/results/random_forest_results.csv', index=False)
    print('Results saved to StudentPerformance/results/random_forest_results.csv')

except Exception as e:
    print('Error running Random Forest evaluation:', e)


SyntaxError: unterminated string literal (detected at line 29) (2672413607.py, line 29)

In [12]:
# Train (if needed), evaluate and save Random Forest results
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

try:
    # Ensure training data and preprocessor exist
    if 'X_train' not in globals() or 'y_train' not in globals():
        raise NameError('Training data not found. Run preprocessing cells first.')

    if 'rf_grid_search' not in globals():
        rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])
        param_grid = {
            'classifier__n_estimators': [100],
            'classifier__max_depth': [None, 10]
        }
        rf_grid_search = GridSearchCV(rf_pipeline, param_grid, scoring='f1_macro', cv=5, n_jobs=-1)
        print('Fitting Random Forest grid search (this may take a while)...')
        rf_grid_search.fit(X_train, y_train)

    rf_best = rf_grid_search.best_estimator_
    rf_y_pred = rf_best.predict(X_test)

    # Metrics
    rf_accuracy = accuracy_score(y_test, rf_y_pred)
    rf_precision = precision_score(y_test, rf_y_pred, average='macro')
    rf_recall = recall_score(y_test, rf_y_pred, average='macro')
    rf_f1 = f1_score(y_test, rf_y_pred, average='macro')

    print('
Random Forest - Test Metrics:')
    print(f"Accuracy: {rf_accuracy:.4f}")
    print(f"Precision (macro): {rf_precision:.4f}")
    print(f"Recall (macro): {rf_recall:.4f}")
    print(f"F1-score (macro): {rf_f1:.4f}")

    print('
Random Forest - Classification Report:')
    print(classification_report(y_test, rf_y_pred, target_names=[str(c) for c in label_enc.classes_]))

    # Confusion matrix
    rf_cm = confusion_matrix(y_test, rf_y_pred)
    os.makedirs('StudentPerformance/visualizations', exist_ok=True)
    plt.figure(figsize=(7,5))
    sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=label_enc.classes_, yticklabels=label_enc.classes_)
    plt.title('Confusion Matrix - Random Forest')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig('StudentPerformance/visualizations/random_forest_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Save results
    rf_results_df = pd.DataFrame({
        'Model': ['Random Forest'],
        'Best_Params': [str(rf_grid_search.best_params_)],
        'CV_F1_Macro': [rf_grid_search.best_score_],
        'Test_Accuracy': [rf_accuracy],
        'Test_Precision_Macro': [rf_precision],
        'Test_Recall_Macro': [rf_recall],
        'Test_F1_Macro': [rf_f1]
    })
    os.makedirs('StudentPerformance/results', exist_ok=True)
    rf_results_df.to_csv('StudentPerformance/results/random_forest_results.csv', index=False)
    print('Results saved to StudentPerformance/results/random_forest_results.csv')

except Exception as e:
    print('Error running Random Forest evaluation:', e)


SyntaxError: unterminated string literal (detected at line 29) (2672413607.py, line 29)

In [13]:
# End-to-end Random Forest (self-contained)
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# load data
data_path = 'StudentPerformance/data/Student_performance_data _.csv'
df = pd.read_csv(data_path)

# prepare
target_col = 'GradeClass'
drop_cols = ['StudentID'] if 'StudentID' in df.columns else []
X = df.drop(columns=drop_cols + [target_col])
y = df[target_col]
label_enc = LabelEncoder()
y_encoded = label_enc.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

numeric_features = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object','category','bool']).columns.tolist()

numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, categorical_features)])

pipeline = Pipeline([('preprocessor', preprocessor), ('feature_selection', SelectKBest(score_func=f_classif, k='all')), ('classifier', RandomForestClassifier(random_state=42))])

param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

gs = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
print('Fitting RF GridSearch...')
gs.fit(X_train, y_train)
print('Best params:', gs.best_params_)

best = gs.best_estimator_
rf_pred = best.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred, average='macro')
rf_recall = recall_score(y_test, rf_pred, average='macro')
rf_f1 = f1_score(y_test, rf_pred, average='macro')
rf_cm = confusion_matrix(y_test, rf_pred)

# save visualization
os.makedirs('StudentPerformance/visualizations', exist_ok=True)
plt.figure(figsize=(7,5))
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Greens', xticklabels=label_enc.classes_, yticklabels=label_enc.classes_)
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('StudentPerformance/visualizations/random_forest_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

# save results
os.makedirs('StudentPerformance/results', exist_ok=True)
results_df = pd.DataFrame({
    'Model': ['Random Forest'],
    'Best_Params': [str(gs.best_params_)],
    'CV_F1_Macro': [gs.best_score_],
    'Test_Accuracy': [rf_accuracy],
    'Test_Precision_Macro': [rf_precision],
    'Test_Recall_Macro': [rf_recall],
    'Test_F1_Macro': [rf_f1]
})
results_df.to_csv('StudentPerformance/results/random_forest_results.csv', index=False)
print('Saved random forest results and visualization')


Fitting RF GridSearch...


Best params: {'classifier__max_depth': None, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 50}


Saved random forest results and visualization
